# ChemEcho: Sonifying Molecular IR Spectra

**Course project — EPFL**  

---

## 1. Introduction

ChemEcho is a Python package that translates infrared (IR) absorption spectra into MIDI music. Each detected absorption peak in a molecule's IR spectrum becomes a musical note, with four independent physical properties of the peak encoded as musical dimensions:

| Spectral property | Musical dimension | Physical meaning |
|---|---|---|
| Wavenumber position | Pitch | Type of bond vibration (C–H, C=O, O–H, …) |
| Absorption prominence | Velocity (loudness) | Strength of the vibration / dipole change |
| Peak FWHM width | Note duration | Vibrational mode lifetime / IVR coupling |
| Gap between peaks | Note interval | Distribution of peaks across the spectrum |
| Molecular weight | Tempo (BPM) | Kinetic theory: lighter molecules move faster |
| Molecular weight | Instrument | Heavier molecules → lower-pitched instruments |

The result is a short piece of music that is unique to each molecule and encodes its spectroscopic identity in an audible form.

---

## 2. Scientific Background

### 2.1 Infrared Spectroscopy

An IR spectrum records how much infrared light a molecule absorbs at each wavenumber (cm⁻¹). Each absorption peak corresponds to a specific bond vibration mode:

- **~3000 cm⁻¹**: C–H stretching  
- **~1700 cm⁻¹**: C=O stretching (strong, characteristic of ketones, aldehydes, esters)  
- **~3300 cm⁻¹**: O–H stretching (broad due to hydrogen bonding)  
- **400–1500 cm⁻¹**: fingerprint region (complex, molecule-specific)

The conventional x-axis runs from high wavenumber (4000 cm⁻¹) on the left to low wavenumber (400 cm⁻¹) on the right, so the functional-group region appears first.

### 2.2 Motivation for Sonification

IR-to-music conversion has been explored in recent literature as a tool for:
- Making spectroscopic data accessible to visually impaired scientists
- Building intuition for spectral patterns through a different sensory channel
- Science communication and outreach

Key references informing the design of ChemEcho:

1. **Kim & Heller** (arXiv 2601.02652, 2026) — FWHM → note duration encoding, peak width as a proxy for intramolecular vibrational redistribution (IVR)
2. **Osterroth et al.** (J. Chem. Ed. 97, 703, 2020) — wavenumber-to-pitch linear mapping across the IR range
3. **Mahjour et al.** (Digital Discovery, 2023) — multi-parameter sonification encoding independent spectral dimensions
4. **Hermann & Casini**, Sonification Handbook Ch. 15 (2011) — gap-proportional time intervals as the most direct parameter mapping

---

## 3. Package Structure

```
src/chemecho/
├── get_spectrum.py        # NIST data fetching and IR graph
├── spectrum_utils.py      # Unit conversion (yunits, xunits)
├── musification_2.0.py    # Peak detection and MIDI generation
└── streamlit_app.py       # Interactive web application
```

Data flow:

```
Molecule name
    → NIST WebBook (nistchempy)
    → JCAMP-DX file (raw spectrum)
    → unit normalisation (spectrum_utils)
    → transmittance vs. wavenumber arrays
    → peak detection (scipy.signal.find_peaks)
    → parameter mapping
    → MIDI file (midiutil)
```

---

## 4. Setup

In [ ]:
import sys
import importlib.util
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import nistchempy as nist

# Add src/chemecho to path
pkg = Path('..') / 'src' / 'chemecho'
sys.path.insert(0, str(pkg))

from get_spectrum import extract_spectrum_data, ir_graph, from_df_to_csv
from spectrum_utils import classify_yunits, to_transmittance, to_wavenumber, YUNITS_PRIORITY

# musification_2.0.py cannot be imported directly (dots in filename)
_mus_path = pkg / 'musification_2.0.py'
_spec = importlib.util.spec_from_file_location('musification', _mus_path)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

mw_to_bpm = _mod.mw_to_bpm
wavenumber_to_midi = _mod.wavenumber_to_midi
peak_detection = _mod.peak_detection
molecular_music = _mod.molecular_music
molecular_weight_to_sound_code = _mod.molecular_weight_to_sound_code

print('Setup complete.')

---

## 5. Fetching IR Data from NIST

ChemEcho fetches spectra from the [NIST WebBook](https://webbook.nist.gov/) using the `nistchempy` library. Spectra are stored in JCAMP-DX (`.jdx`) format and may come in different units depending on the instrument and lab that recorded them.

The `extract_spectrum_data` function:
1. Calls `compound.get_ir_spectra()` to download available IR spectra
2. Selects the best spectrum based on unit priority: transmittance > absorbance > molar absorptivity
3. Converts x-units to wavenumber (cm⁻¹) and y-units to transmittance (%)
4. Returns `(wavenumbers, transmittances)` as two Python lists

In [ ]:
# Fetch acetone (CAS 67-64-1) as a running example
compound = nist.get_compound('67-64-1')
print(f'Name:              {compound.name}')
print(f'Molecular formula: {compound.mol_form}')
print(f'Molecular weight:  {compound.mol_weight} Da')

In [ ]:
wavenumbers, transmittances = extract_spectrum_data(compound)

print(f'Number of data points: {len(wavenumbers)}')
print(f'Wavenumber range:      {min(wavenumbers):.0f} – {max(wavenumbers):.0f} cm⁻¹')
print(f'Transmittance range:   {min(transmittances):.1f} – {max(transmittances):.1f} %')

### 5.1 Unit Normalisation (`spectrum_utils.py`)

NIST spectra arrive in varied units. `spectrum_utils` handles three cases for the y-axis and two for the x-axis:

**Y-axis (yunits):**
- `TRANSMITTANCE` — kept as-is (or rescaled from 0–1 to 0–100%)
- `ABSORBANCE` — converted via $T = 10^{-A} \times 100$
- `MOLAR_ABSORPTIVITY` — treated as absorbance

**X-axis (xunits):**
- `1/CM` or `CM-1` — kept as-is
- `MICROMETERS` — converted via $\tilde{\nu} = 10000 / \lambda_{\mu m}$

In [ ]:
# Unit classification examples
test_cases = ['TRANSMITTANCE', 'Absorbance', 'MICROMOL/MOL', 'HERTZ']
for label in test_cases:
    print(f'  classify_yunits({label!r:25s}) → {classify_yunits(label)}')

### 5.2 Visualising the Spectrum

In [ ]:
matplotlib.use('inline')  # switch to interactive backend for notebook
import importlib; import matplotlib; importlib.reload(matplotlib)
%matplotlib inline

fig, ax = ir_graph((wavenumbers, transmittances), compound.name)
plt.tight_layout()
plt.show()

---

## 6. Peak Detection

The `peak_detection` function identifies absorption dips — local minima in the transmittance curve — using `scipy.signal.find_peaks` applied to the *inverted* transmittance signal.

Two parameters control sensitivity:
- **`min_prominence`**: minimum absorption depth (%) to count as a real peak. Filters baseline noise.
- **`min_width_pts`**: minimum width in data points. Filters single-sample spikes.

For each detected peak, the function returns a tuple `(wavenumber, prominence, fwhm_width_cm⁻¹)`. The FWHM width is converted from data-point units to cm⁻¹ using the average point spacing.

In [ ]:
peaks = peak_detection(wavenumbers, transmittances, min_prominence=15.0, min_width_pts=3)

print(f'Peaks detected: {len(peaks)}\n')
print(f'{"Wavenumber (cm⁻¹)":>20} {"Prominence (%)"):>18} {"FWHM (cm⁻¹)":>14}')
print('-' * 55)
for wn, prom, width in peaks:
    print(f'{wn:>20.1f} {prom:>18.1f} {width:>14.1f}')

In [ ]:
# Overlay detected peaks on the spectrum
fig, ax = ir_graph((wavenumbers, transmittances), compound.name)

peak_wns   = [p[0] for p in peaks]
peak_trans = [transmittances[wavenumbers.index(wn)] for wn in peak_wns]

ax.scatter(peak_wns, peak_trans, color='red', zorder=5, s=40, label='Detected peaks')
ax.legend()
plt.tight_layout()
plt.show()

---

## 7. Parameter Mapping

### 7.1 Wavenumber → Pitch

Each peak's wavenumber position maps linearly to a MIDI pitch number:

$$\text{MIDI} = 40 + \frac{\tilde{\nu} - 400}{4000 - 400} \times 44 \quad \in [40, 84]$$

This places 400 cm⁻¹ at E2 (MIDI 40) and 4000 cm⁻¹ at C6 (MIDI 84), consistent with Osterroth et al. (2020).

In [ ]:
import numpy as np

test_wns = [400, 1000, 1715, 3000, 4000]  # fingerprint, C-C, C=O, C-H, upper limit
labels   = ['400 cm⁻¹ (lower bound)', '1000 cm⁻¹ (fingerprint)', 
             '1715 cm⁻¹ (C=O stretch)', '3000 cm⁻¹ (C-H stretch)', '4000 cm⁻¹ (upper bound)']

for wn, label in zip(test_wns, labels):
    midi = wavenumber_to_midi(wn)
    print(f'  {label:35s} → MIDI {midi}')

### 7.2 Molecular Weight → Tempo

Tempo (BPM) is derived from molecular weight using a kinetic-theory analogy. The RMS velocity of a molecule scales as $v_{rms} \propto 1/\sqrt{M}$, so lighter molecules are assigned a faster tempo:

$$\text{BPM} = 170 - \frac{\sqrt{M} - \sqrt{18}}{\sqrt{600} - \sqrt{18}} \times 110 \quad \in [60, 170]$$

In [ ]:
molecules = [
    ('Water',     18.02),
    ('Ethanol',   46.07),
    ('Acetone',   58.08),
    ('Caffeine',  194.19),
    ('Aspirin',   180.16),
    ('Ibuprofen', 206.29),
]

print(f'{"Molecule":>12} {"MW (Da)":>10} {"BPM":>6}')
print('-' * 32)
for name, mw in molecules:
    print(f'{name:>12} {mw:>10.2f} {mw_to_bpm(mw):>6}')

### 7.3 Molecular Weight → Instrument

A General MIDI instrument is selected based on molecular weight, following the analogy that heavier objects vibrate at lower frequencies:

In [ ]:
instrument_names = {
    40: 'Violin',
    71: 'Clarinet',
    42: 'Cello',
    66: 'Tenor Saxophone',
    43: 'Contrabass',
}

print(f'{"Molecule":>12} {"MW (Da)":>10} {"Instrument":>18}')
print('-' * 44)
for name, mw in molecules:

    class _FakeCmpd:
        mol_weight = mw

    code = molecular_weight_to_sound_code(_FakeCmpd())
    print(f'{name:>12} {mw:>10.2f} {instrument_names[code]:>18}')

### 7.4 Note Interval — Wavenumber Gap Proportional

Rather than spacing all notes equally, ChemEcho encodes the *distribution* of peaks across the spectrum as rhythm. The gap in wavenumber between consecutive peaks maps to the time interval between notes:

$$\text{raw interval}_i = 0.5 + \frac{\Delta\tilde{\nu}_i - \Delta\tilde{\nu}_{\min}}{\Delta\tilde{\nu}_{\max} - \Delta\tilde{\nu}_{\min}} \times 3.5$$

Intervals are normalised so their sum equals 30 beats (total piece duration). This means two C–H peaks close together at 3000–3050 cm⁻¹ play in rapid succession, while the large empty region between the functional-group and fingerprint regions produces an audible pause.

### 7.5 Note Duration — FWHM-Based

Peak width (FWHM in cm⁻¹) maps to note duration independently of the note interval:

$$\text{duration} = 0.5 + \frac{w - w_{\min}}{w_{\max} - w_{\min}} \times 1.5 \quad \in [0.5, 2.0] \text{ beats}$$

In IR spectroscopy, peak width reflects the vibrational mode lifetime: broader peaks indicate faster intramolecular vibrational redistribution (IVR). A sharp C≡N stretch plays as a short crisp note; a broad O–H stretch (with hydrogen bonding) plays as a long sustained note (Kim & Heller 2026).

In [ ]:
# Show the full parameter mapping for acetone's peaks
wns       = [p[0] for p in peaks]
gaps      = [wns[i] - wns[i+1] for i in range(len(wns)-1)] + [0]
all_widths = [p[2] for p in peaks]
min_w, max_w = min(all_widths), max(all_widths)
width_range  = max(max_w - min_w, 1.0)

print(f'{"cm⁻¹":>8} {"Prom":>7} {"FWHM":>7} {"MIDI":>6} {"Vel":>5} {"Dur (b)":>9} {"Gap":>8}')
print('-' * 58)
for i, (wn, prom, width) in enumerate(peaks):
    midi     = wavenumber_to_midi(wn)
    vel      = int(max(45, min(110, 45 + (prom / 100) * 65)))
    wr       = (width - min_w) / width_range
    dur      = 0.5 + wr * 1.5
    gap      = gaps[i]
    print(f'{wn:>8.1f} {prom:>7.1f} {width:>7.1f} {midi:>6} {vel:>5} {dur:>9.2f} {gap:>8.1f}')

---

## 8. Generating MIDI

The `molecular_music` function orchestrates all steps and writes a `.mid` file using the `midiutil` library. The file contains two tracks:

- **Track 0**: melody — one note per detected peak, using the four-dimensional parameter mapping above
- **Track 1** (optional): percussion on MIDI channel 9 — kick drum (MIDI 36) every 500 cm⁻¹, hi-hat (MIDI 42) every 100 cm⁻¹, acting as a spectral ruler across the 30-beat duration

In [ ]:
import tempfile, os

with tempfile.NamedTemporaryFile(suffix='.mid', delete=False) as tmp:
    midi_path = tmp.name

molecular_music(compound, midi_path, data=(wavenumbers, transmittances), drum_track=True)

size_kb = os.path.getsize(midi_path) / 1024
print(f'MIDI file written: {midi_path}')
print(f'File size:         {size_kb:.1f} KB')
print(f'Tempo:             {mw_to_bpm(compound.mol_weight)} BPM')
print(f'Peaks (notes):     {len(peaks)}')

os.remove(midi_path)

---

## 9. Comparison Across Molecules

The table below shows how the sonification parameters differ across three molecules with distinct chemistry, illustrating that each molecule produces a genuinely different piece of music.

In [ ]:
cas_list = [
    ('Acetone',  '67-64-1'),
    ('Ethanol',  '64-17-5'),
    ('Caffeine', '58-08-2'),
]

rows = []
for mol_name, cas in cas_list:
    cmpd = nist.get_compound(cas)
    wns_i, trans_i = extract_spectrum_data(cmpd)
    pks = peak_detection(wns_i, trans_i)
    rows.append({
        'Molecule': mol_name,
        'MW (Da)': cmpd.mol_weight,
        'BPM': mw_to_bpm(cmpd.mol_weight),
        'Peaks': len(pks),
        'Pitch range': f'{wavenumber_to_midi(min(p[0] for p in pks))}–{wavenumber_to_midi(max(p[0] for p in pks))}',
        'Max velocity': max(int(max(45, min(110, 45 + (p[1]/100)*65))) for p in pks),
    })

pd.DataFrame(rows).set_index('Molecule')

---

## 10. Interactive Application

ChemEcho is deployed as a Streamlit web application (`src/chemecho/streamlit_app.py`). The app allows any user to:

1. Enter a molecule name (e.g. *aspirin*, *dopamine*, *quinine*)
2. View the 2D molecular structure (fetched from PubChem)
3. View the IR spectrum plot
4. Play the generated music directly in the browser via an embedded MIDI player
5. Download the spectrum as CSV or the music as a `.mid` file

To launch the app:

```bash
conda activate chemecho
streamlit run src/chemecho/streamlit_app.py
```

---

## 11. Conclusion

ChemEcho demonstrates that IR spectra can be translated into music in a scientifically principled way, where every musical dimension carries independent physical meaning:

- **Pitch** encodes bond type (the most chemically informative dimension of an IR spectrum)
- **Rhythm** encodes spectral density and the distribution of peaks across the spectrum
- **Loudness** encodes absorption strength
- **Duration** encodes peak sharpness and vibrational mode coupling
- **Tempo and instrument** encode molecular size

The design is grounded in published IR sonification literature and produces output that is distinct and recognisable for molecules with different functional groups. Future directions include harmonics-based chord generation for overlapping peaks and real-time synthesis (rather than MIDI file output).